In [33]:
!pip install --upgrade yfinance


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\felip\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [34]:
import requests, os
from datetime import date
import pandas as pd

In [35]:
# --- Configuração ---
URL = "https://companiesmarketcap.com/largest-companies-by-revenue/?download=csv"
OUTPUT_DIR = "../data/raw/"
FILE_PREFIX = "top_companies_revenue"
# --------------------

today = date.today().strftime("%Y%m%d")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_{today}.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

with open(OUTPUT_PATH, "wb") as f:
    f.write(response.content)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Tamanho: {len(response.content):,} bytes")


Arquivo salvo em: ../data/raw/top_companies_revenue_20260906.csv
Tamanho: 765,376 bytes


In [36]:
import glob
files = sorted(glob.glob("../data/raw/top_companies_revenue_*.csv"))
raw_data_file = files[-1]

print(f"Arquivo mais recente encontrado: {raw_data_file}")

import pandas as pd
df = pd.read_csv(raw_data_file)

std_columns = {
    "Rank": "rank",
    "Name": "company_name",
    "Symbol": "ticker_symbol",
    "revenue_ttm": "revenue_ttm",
    "price (USD)": "price_usd",
    "country": "company_country",
}
df = df.rename(columns=std_columns)


from bs4 import BeautifulSoup
def clean_html(text):
    if pd.isna(text):
        return text
    return BeautifulSoup(text, "html.parser").get_text()

df["company_name"] = df["company_name"].apply(clean_html)

#check for non unique tickers and print the ones that are not unique 
non_unique_tickers = df[df.duplicated(subset=["ticker_symbol"], keep=False)]
if not non_unique_tickers.empty:
    print("Non-unique tickers found:")
    print(non_unique_tickers[["ticker_symbol", "company_name"]])

display(df.head(10))

Arquivo mais recente encontrado: ../data/raw\top_companies_revenue_20260906.csv


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country
0,1,Amazon,AMZN,742776000000,258.510000,United States
1,2,Walmart,WMT,725305000000,107.140000,United States
2,3,Saudi Aramco,2222.SR,459047318045,6.913820,Saudi Arabia
3,4,Apple,AAPL,451442000000,319.970000,United States
4,5,UnitedHealth,UNH,449713000000,397.140000,United States
5,6,Sinopec,600028.SS,444749587530,0.817898,China
6,7,Alphabet (Google),GOOG,422499000000,335.310000,United States
7,8,CVS Health,CVS,407905000000,96.740000,United States
8,9,PetroChina,0857.HK,407253551390,1.295730,China
9,10,McKesson,MCK,403430000000,907.990000,United States


In [37]:
import os
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from curl_cffi import requests as cffi_requests
import time
import pandas as pd

OUTPUT_INTERIM_PATH = "../data/interim/01b_data_after_yfinance.csv"

COLUNAS_YF = [
    "ticker_symbol", "website", "longBusinessSummary",
    "sector", "industry", "country", "city", "yf_status"
]


import threading
import time

# Um objeto mutável simples pra guardar "até quando estamos pausados".
# Não pode ser uma variável comum (int/float solto), porque threads
# não compartilham escopo local -- precisa ser algo em memória
# acessível por todas (aqui, uma lista com 1 elemento, ou um objeto).
estado_pausa = {"bloqueado_ate": 0}

# Lock evita que duas threads, ao mesmo tempo, tentem escrever
# em estado_pausa["bloqueado_ate"] e uma sobrescreva a outra
# de forma inconsistente (condição de corrida clássica).
lock_pausa = threading.Lock()

def aguardar_se_pausado():
    # Cada thread, antes de chamar o Yahoo, primeiro checa se
    # "ainda estamos no período de pausa" definido por QUALQUER
    # outra thread que tenha detectado rate limit.
    agora = time.time()
    se_bloqueado = estado_pausa["bloqueado_ate"] > agora
    if se_bloqueado:
        tempo_restante = estado_pausa["bloqueado_ate"] - agora
        time.sleep(tempo_restante)
        # Após dormir, a thread segue para tentar a chamada normalmente

def ativar_pausa_global(segundos=60):
    with lock_pausa:
        # Usamos max() porque, se 5 threads baterem rate limit quase
        # ao mesmo tempo, não queremos que a 5ª thread "encurte"
        # a pausa que a 1ª já tinha definido -- sempre estende, nunca reduz.
        novo_horario = time.time() + segundos
        estado_pausa["bloqueado_ate"] = max(estado_pausa["bloqueado_ate"], novo_horario)


def criar_sessao():
    # Função separada pra podermos "recriar do zero" quando necessário,
    # em vez de reaproveitar uma sessao_global que pode ter ficado inválida
    return cffi_requests.Session(impersonate="chrome")

sessao_global = criar_sessao()
lock_sessao = threading.Lock()

def recriar_sessao_global():
    global sessao_global
    with lock_sessao:
        # Lock aqui evita que 10 threads recriem a sessão ao mesmo
        # tempo desperdiçando handshakes -- só a primeira que perceber
        # o problema recria; as outras reaproveitam a nova
        sessao_global = criar_sessao()

def get_yfinance_info(ticker):
    aguardar_se_pausado()

    try:
        info = yf.Ticker(ticker, session=sessao_global).info
        return {
            "ticker_symbol": ticker,
            "website": info.get("website"),
            "longBusinessSummary": info.get("longBusinessSummary"),
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "country": info.get("country"),
            "city": info.get("city"),
            "yf_status": "success"
        }
    except Exception as e:
        mensagem = str(e)

        if "Too Many Requests" in mensagem or "Rate limited" in mensagem:
            ativar_pausa_global(segundos=60)
            aguardar_se_pausado()
            try:
                info = yf.Ticker(ticker, session=sessao_global).info
                return {"ticker_symbol": ticker, "website": info.get("website"),
                        "longBusinessSummary": info.get("longBusinessSummary"),
                        "sector": info.get("sector"), "industry": info.get("industry"),
                        "country": info.get("country"), "city": info.get("city"),
                        "yf_status": "success"}
            except Exception as e2:
                mensagem = str(e2)

        if "Not Found" in mensagem or "Quote not found" in mensagem:
            return {"ticker_symbol": ticker, "website": None, "longBusinessSummary": None,
                    "sector": None, "industry": None, "country": None, "city": None,
                    "yf_status": "not_found"}

        # --- Caso novo: 401 Unauthorized -> problema de sessão, não do ticker ---
        if "Unauthorized" in mensagem or "401" in mensagem:
            print(f"Sessão inválida detectada (ticker {ticker}), recriando sessão...")
            recriar_sessao_global()
            aguardar_se_pausado()
            try:
                info = yf.Ticker(ticker, session=sessao_global).info
                return {"ticker_symbol": ticker, "website": info.get("website"),
                        "longBusinessSummary": info.get("longBusinessSummary"),
                        "sector": info.get("sector"), "industry": info.get("industry"),
                        "country": info.get("country"), "city": info.get("city"),
                        "yf_status": "success"}
            except Exception as e2:
                mensagem = str(e2)
                # Se falhar de novo mesmo com sessão nova, marcamos como
                # "error" (não "not_found"), pois é diferente/temporário --
                # vale reprocessar na PRÓXIMA execução do notebook
                return {"ticker_symbol": ticker, "website": None, "longBusinessSummary": None,
                        "sector": None, "industry": None, "country": None, "city": None,
                        "yf_status": f"error: {mensagem}"}

        print(f"Error fetching data for {ticker}: {mensagem}")
        return {"ticker_symbol": ticker, "website": None, "longBusinessSummary": None,
                "sector": None, "industry": None, "country": None, "city": None,
                "yf_status": f"error: {mensagem}"}

# --- 1. Retomar de execução anterior, se existir ---
if os.path.exists(OUTPUT_INTERIM_PATH):
    df_existente = pd.read_csv(OUTPUT_INTERIM_PATH)
    colunas_extra = [c for c in COLUNAS_YF if c != "ticker_symbol"]
    df = pd.merge(
        df.drop(columns=[c for c in colunas_extra if c in df.columns]),
        df_existente[COLUNAS_YF],
        on="ticker_symbol",
        how="left"
    )
else:
    for col in COLUNAS_YF:
        if col not in df.columns:
            df[col] = None

# --- 2. Filtrar só quem falta ---
mascara_pendente = df["yf_status"].iloc[::-1] != "success" # começar de tras pra frente, pra não perder o que já foi feito
df_to_process = df.loc[mascara_pendente, "ticker_symbol"]
print(f"Total: {len(df)} | Já ok: {(~mascara_pendente).sum()} | Pendentes: {len(df_to_process)}")

# --- 3. Loop em chunks (igual ao seu, sem mudanças) ---
chunck_size = 100
wait_time = 5

for i in tqdm(range(0, len(df_to_process), chunck_size), desc="Processing chunks"):
    chunk = df_to_process[i:i + chunck_size]
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(executor.map(get_yfinance_info, chunk))

    time.sleep(wait_time)

    df_yfinance = pd.DataFrame(results)
    df_yfinance.set_index("ticker_symbol", inplace=True)

    df.set_index("ticker_symbol", inplace=True)
    df.update(df_yfinance)
    df.reset_index(inplace=True)

    df.to_csv(OUTPUT_INTERIM_PATH, index=False)

display(df.head(10))

Total: 11231 | Já ok: 10599 | Pendentes: 632


Processing chunks:   0%|          | 0/7 [00:00<?, ?it/s]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: IHC.AE"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: QVCGA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: JDEP.AS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 7205.T"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: XEKT.MC"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 0489.HK"}}}
HTTP Error 404: {"quoteSummary":

Error fetching data for nan: 'float' object has no attribute 'upper'


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PWSA.KH"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FRGE"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LSB"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: JPASSOCIAT.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PAS.KH"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MBCN"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: RECSI.OL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quot

,ticker_symbol,rank,company_name,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
0,AMZN,1,Amazon,742776000000,258.510000,United States,https://www.amazon.com,"Amazon.com, Inc. engages in the retail sale of...",Consumer Cyclical,Internet Retail,United States,Seattle,success
1,WMT,2,Walmart,725305000000,107.140000,United States,https://www.stock.walmart.com,Walmart Inc. engages in the operation of retai...,Consumer Defensive,Discount Stores,United States,Bentonville,success
2,2222.SR,3,Saudi Aramco,459047318045,6.913820,Saudi Arabia,https://www.aramco.com,Saudi Arabian Oil Company operates as an integ...,Energy,Oil & Gas Integrated,Saudi Arabia,Dhahran,success
3,AAPL,4,Apple,451442000000,319.970000,United States,https://www.apple.com,"Apple Inc. designs, manufactures, and markets ...",Technology,Consumer Electronics,United States,Cupertino,success
4,UNH,5,UnitedHealth,449713000000,397.140000,United States,https://www.unitedhealthgroup.com,UnitedHealth Group Incorporated operates as a ...,Healthcare,Healthcare Plans,United States,Eden Prairie,success
5,600028.SS,6,Sinopec,444749587530,0.817898,China,https://www.sinopec.com/listco,"China Petroleum & Chemical Corporation, an ene...",Energy,Oil & Gas Integrated,China,Beijing,success
6,GOOG,7,Alphabet (Google),422499000000,335.310000,United States,https://abc.xyz,Alphabet Inc. offers various products and plat...,Communication Services,Internet Content & Information,United States,Mountain View,success
7,CVS,8,CVS Health,407905000000,96.740000,United States,https://www.cvshealth.com,CVS Health Corporation provides health solutio...,Healthcare,Healthcare Plans,United States,Woonsocket,success
8,0857.HK,9,PetroChina,407253551390,1.295730,China,https://www.petrochina.com.cn,"PetroChina Company Limited, together with its ...",Energy,Oil & Gas Integrated,China,Beijing,success
9,MCK,10,McKesson,403430000000,907.990000,United States,https://www.mckesson.com,McKesson Corporation provides healthcare servi...,Healthcare,Medical Distribution,United States,Irving,success


In [40]:
import pandas as pd

check_df = pd.read_csv("../data/interim/01b_data_after_yfinance.csv")


# Máscara 1: linhas onde o yf_status não foi sucesso
sucesso = check_df["yf_status"] == "success"

# Máscara 2: website ausente (NaN real) ou string vazia
website_vazio = check_df["website"].isna() | (check_df["website"] == "")

# Combina as duas condições com & (E lógico bit a bit — para Series,
# use sempre & / | / ~ em vez de and/or/not, que não funcionam elemento a elemento)
mask = sucesso & website_vazio

check_df.loc[mask, "yf_status"] = "error: website not found"

# opcional: conferir quantas linhas foram afetadas
print(mask.sum())

check_df.to_csv("../data/interim/01b_data_after_yfinance.csv", index=False)


46


### Usando yfinance search pra resolver casos em que o ticker não bateu

In [39]:
import yfinance as yf
import pandas as pd
from tqdm import tqdm
import time


def get_yfinance_info_by_name(name):
    
    try:
        busca = yf.Search(name)
        if not busca.quotes:
            return {"name": name, "ticker_symbol": None, "website": None, "yf_status": "not_found_yf_search"}
        
        # Pega o primeiro resultado da busca
        resultado = busca.quotes[0]
        ticker = resultado.get("symbol")
        info = yf.Ticker(ticker).info
        
        return {
            "name": name,
            "ticker_symbol": ticker,
            "website": info.get("website"),
            "longBusinessSummary": info.get("longBusinessSummary"),
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "country": info.get("country"),
            "city": info.get("city"),
            "yf_status": "success"
        }
    except Exception as e:
        mensagem = str(e)
        if "Not Found" in mensagem or "Quote not found" in mensagem:
            return {"name": name, "ticker_symbol": None, "website": None, "yf_status": "not_found"}
        else:
            return {"name": name, "ticker_symbol": None, "website": None, "yf_status": f"error: {mensagem}"}

missing_websites_raw = pd.read_csv("../data/interim/01b_data_after_yfinance.csv")

missing_websites = missing_websites_raw[missing_websites_raw["yf_status"] != "success"]

i = 0

for company in tqdm(missing_websites["company_name"], desc="Processing companies"):  # Limit to first 10 companies for testing
    info = get_yfinance_info_by_name(company)
    missing_websites.loc[
        missing_websites["company_name"] == company, 
        ["ticker_symbol", "website", "longBusinessSummary", "sector", "industry", "country", "city", "yf_status"]] = [
        info.get("ticker_symbol"),
        info.get("website"),
        info.get("longBusinessSummary"),
        info.get("sector"),
        info.get("industry"),
        info.get("country"),
        info.get("city"),
        info.get("yf_status")
    ]
    i += 1
    if i % 100 == 0:
        print(f"Processed {i} companies")
        wait_time = 10
        time.sleep(wait_time)

#salvar de forma incremental, pra não perder tudo se der erro no meio
missing_websites_raw.update(missing_websites)

missing_websites_raw.to_csv("../data/interim/01b_data_after_yfinance.csv", index=False)

Processing companies:   0%|          | 0/632 [00:00<?, ?it/s]

Processing companies:  16%|█▌        | 99/632 [01:02<05:14,  1.69it/s]

Processed 100 companies


Processing companies:  31%|███▏      | 199/632 [02:09<03:46,  1.91it/s]

Processed 200 companies


Processing companies:  47%|████▋     | 299/632 [03:29<03:12,  1.73it/s]

... (4 outputs left out)

Processed 500 companies


Processing companies:  95%|█████████▍| 599/632 [06:10<00:09,  3.33it/s]

Processed 600 companies


Processing companies: 100%|██████████| 632/632 [06:33<00:00,  1.60it/s]
